Batching Simulations
====================

This tutorial will cover how to set up batched simulations.

In [ ]:
!pip install icrn

In [1]:
import jax
import jax.numpy as jnp

from icrn import (
    MassActionReaction,
    many_index_symbols,
    many_rate_constants,
    many_species,
    solve_well_mixed,
)

Suppose that one wants to simulate a batch of dimer systems, each with its own set of rate constants. `icrn` solvers are ordinary [JAX](https://docs.jax.dev/) functions, so an entire batch can be evaluated at once with [`jax.vmap`](https://docs.jax.dev/en/latest/_autosummary/jax.vmap.html). On a GPU the batched simulations run in parallel.

We first build the dimer network: two monomers `M[i]` and `M[j]` reversibly bind to form a dimer `D[i, j]`.

In [2]:
MONOMERS = 10

M, D = many_species("M, D")
k1, k2 = many_rate_constants("k1, k2")
i, j = many_index_symbols("i, j", MONOMERS)

dimer_rxns = [
    MassActionReaction(M[i] + M[j], D[i, j], k1[i, j]),
    MassActionReaction(D[i, j], M[i] + M[j], k2[i, j]),
]

The initial concentrations are shared across the batch, so they keep their ordinary shapes: `M` has shape `(MONOMERS,)` and `D` has shape `(MONOMERS, MONOMERS)`.

The rate constants differ between batch elements. We give each rate-constant array a leading batch dimension of size `BATCH_SIZE`; this is the axis we will map over.

In [3]:
BATCH_SIZE = 8

conc_vals = {
    M: jnp.ones(MONOMERS),
    D: jnp.zeros((MONOMERS, MONOMERS)),
}

k1_arr = jax.random.uniform(
    jax.random.key(12), (BATCH_SIZE, MONOMERS, MONOMERS)
)
k2_arr = jax.random.uniform(
    jax.random.key(13), (BATCH_SIZE, MONOMERS, MONOMERS)
)

batched_rate_vals = {k1: k1_arr, k2: k2_arr}

`jax.vmap` turns a function that solves a single system into one that solves a whole batch. We wrap `solve_well_mixed` in a small helper that fixes the reactions, times, and step size, exposing only the concentrations and rate constants. We then map over the rate-constant argument with `in_axes=(None, 0)`: the concentrations are shared (`None`) while the rate constants vary along their leading axis (`0`).

In [4]:
times = jnp.array([10.0])
dt = 1e-4


def solve_dimer(conc_vals, rate_constant_vals):
    return solve_well_mixed(
        dimer_rxns,
        conc_vals,
        rate_constant_vals,
        times,
        dt,
        reaction_solver="Euler",
    )


batched_solve = jax.vmap(solve_dimer, in_axes=(None, 0))

batched_concs = batched_solve(conc_vals, batched_rate_vals)

Every species in the result now carries a leading batch dimension, followed by the time axis and the species' own index axes.

In [5]:
print(batched_concs[M].shape)  # (BATCH_SIZE, len(times), MONOMERS)
print(batched_concs[D].shape)  # (BATCH_SIZE, len(times), MONOMERS, MONOMERS)

(8, 1, 10)
(8, 1, 10, 10)


Mapping over the batch with `jax.vmap` is typically much faster than looping in Python, especially on a GPU. Below we compare the two. Each variant is called once first so that [just-in-time compilation](https://docs.jax.dev/en/latest/jit-compilation.html) happens outside the timed region, and we call `jax.block_until_ready` so the asynchronous results are fully computed before timing stops.

In [6]:
%%timeit
jax.block_until_ready(batched_solve(conc_vals, batched_rate_vals))

403 ms ± 6.27 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


For comparison, we solve each system one at a time in a Python loop. We `jit`-compile the single-system solve so the comparison is fair.

In [7]:
single_solve = jax.jit(solve_dimer)

# trace once so compilation is excluded from the timing below
for idx in range(BATCH_SIZE):
    rate_vals = {k1: k1_arr[idx], k2: k2_arr[idx]}
    jax.block_until_ready(single_solve(conc_vals, rate_vals))

In [8]:
%%timeit
for idx in range(BATCH_SIZE):
    rate_vals = {k1: k1_arr[idx], k2: k2_arr[idx]}
    jax.block_until_ready(single_solve(conc_vals, rate_vals))

688 ms ± 25.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


The difference may be more dramatic with a larger `BATCH_SIZE`.